In [ ]:
# CELL 0
# MOUNTING DRIVE

from google.colab import drive
drive.mount('/content/drive')

# ----------------- CHECKING PATH TO AN IMAGE

import os

# insert the name the your Drive folder which contains a shortcut to the project data
my_drive_folder = "APAI_CVDL_shared"

# define root
ROOT = "/content"

In [ ]:
# CELL
# LOADING THE MANIFEST

import os

# insert the name the your Drive folder which contains a shortcut to the project data
my_drive_folder = "APAI_CVDL_shared"

# build the path to the correct manifest file
META_DIR = f"/content/drive/MyDrive/{my_drive_folder}/project/meta"

# build the path to the correct manifest file
manifest_path = f"{META_DIR}/manifest.parquet"

# expected True if path exists
print(os.path.exists(manifest_path))

In [ ]:
# CELL
# DATAFRAME STUFF

import pandas as pd
from sklearn.model_selection import train_test_split

# the entire dataset
df = pd.read_parquet(manifest_path)

# samples selected and "labeled" during AL
df_labeled = df[df["round_added"] != -1].copy()

# training split for student training
df_train_s, df_val_s = train_test_split(
    df_labeled,
    test_size=0.2,
    random_state=7,
    stratify=df_labeled["class"] # preserve natural distribution of the df_labeled for both train and val splits
)

# checks
print(
    f"transfer dataset size: {len(df_labeled)}\n",
    f"student train split size: {len(df_train_s)}\n",
    f"student val split size: {len(df_val_s)}\n",
    f"train pos/neg ratio: {df_train_s['class'].mean()}\n",
    f"val pos/neg ratio: {df_val_s['class'].mean()}\n",
    f"df_labeled pos/neg ratio: {df_labeled['class'].mean()}"
)

In [ ]:
# CELL
# CREATING THE DATASET CLASS

import os
import numpy as np
import torch
from torch.utils.data import Dataset

class StudentDataset(Dataset):
    def __init__(self, df, root, img_col="img_path", mask_col="mask_path"):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.img_col = img_col
        self.mask_col = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # replace slashes and join paths
        img_rel_path = row[self.img_col].replace("\\", "/")
        mask_rel_path = row[self.mask_col].replace("\\", "/")
        img_path  = os.path.join(self.root, str(img_rel_path))
        mask_path = os.path.join(self.root, str(mask_rel_path))

        # load image and mask given path AND convert them to float32
        img = np.load(img_path).astype(np.float32)
        mask = np.load(mask_path).astype(np.float32)

        # from (H, W) to (1, H, W)
        img = torch.from_numpy(img).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask

# ----------- CREATING SPLIT DATASETS

train_ds_s = StudentDataset(df=df_train_s, root=ROOT)
val_ds_s = StudentDataset(df=df_val_s, root=ROOT)

In [ ]:
# CELL
# BUILDING DATALOADERS, IMPLEMENTING OVERSAMPLING

from torch.utils.data import DataLoader
import numpy as np
import torch

# training dataloader
train_loader = DataLoader(
    train_ds_s,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

# validation dataloader
val_loader = DataLoader(
    val_ds_s,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

# ----------------- CHECK (expexted out: (B, 1, 512, 512) float32)

import torch

imgs, masks = next(iter(train_loader))
print("imgs:", imgs.shape, imgs.dtype, "min/max:", imgs.min().item(), imgs.max().item())
print("masks:", masks.shape, masks.dtype, "unique:", torch.unique(masks))

In [ ]:
# CELL
# STUDENT MODEL DEFINITION

import torch
import torch.nn as nn

def double_convolution(in_channels, out_channels, num_groups=8):

    # ensure num_groups divides out_channels for GN
    g = min(num_groups, out_channels)
    while out_channels % g != 0:
        g -= 1

    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.GroupNorm(g, out_channels),
        nn.ReLU(inplace=True),

        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.GroupNorm(g, out_channels),
        nn.ReLU(inplace=True)
    )

class StudentUNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # define encoder convolutions
        self.down_convolution_1 = double_convolution(1, 32)
        self.down_convolution_2 = double_convolution(32, 64)
        self.down_convolution_3 = double_convolution(64, 128)
        self.down_convolution_4 = double_convolution(128, 256)

        # define bottleneck convolutions
        self.down_convolution_bot = double_convolution(256, 512)

        # define encoder downsampling layers
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)

        # define
        self.up_transpose_1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up_transpose_2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_transpose_3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_transpose_4 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)

        # define decoder convolutions
        self.up_convolution_1 = double_convolution(512, 256)
        self.up_convolution_2 = double_convolution(256, 128)
        self.up_convolution_3 = double_convolution(128, 64)
        self.up_convolution_4 = double_convolution(64, 32)

        # define output
        self.out = nn.Conv2d(in_channels=32, out_channels=num_classes, kernel_size=1)

    def forward(self, x):                           # x = (B, 1, 512, 512)
        # encoder
        e1 = self.down_convolution_1(x)             # (B, 32, 512, 512)
        p1 = self.max_pool2d(e1)                    # (B, 32, 256, 256)

        e2 = self.down_convolution_2(p1)            # (B, 64, 256, 256)
        p2 = self.max_pool2d(e2)                    # (B, 64, 128, 128)

        e3 = self.down_convolution_3(p2)            # (B, 128, 128, 128)
        p3 = self.max_pool2d(e3)                    # (B, 128, 64, 64)

        e4 = self.down_convolution_4(p3)            # (B, 256, 64, 64)
        p4 = self.max_pool2d(e4)                    # (B, 256, 32, 32)

        # bottleneck
        bottleneck = self.down_convolution_bot(p4)  # (B, 512, 32, 32)

        # decoder
        t1 = self.up_transpose_1(bottleneck)        # (B, 256, 64, 64)
        c1 = torch.cat([e4, t1], dim=1)             # (B, 512, 64, 64)
        d1 = self.up_convolution_1(c1)              # (B, 256, 64, 64)

        t2 = self.up_transpose_2(d1)                # (B, 128, 128, 128)
        c2 = torch.cat([e3, t2], dim=1)             # (B, 256, 128, 128)
        d2 = self.up_convolution_2(c2)              # (B, 128, 128, 128)

        t3 = self.up_transpose_3(d2)                # (B, 64, 256, 256)
        c3 = torch.cat([e2, t3], dim=1)             # (B, 128, 256, 256)
        d3 = self.up_convolution_3(c3)              # (B, 64, 256, 256)

        t4 = self.up_transpose_4(d3)                # (B, 32, 512, 512)
        c4 = torch.cat([e1, t4], dim=1)             # (B, 64, 512, 512)
        d4 = self.up_convolution_4(c4)              # (B, 32, 512, 512)

        # output
        return self.out(d4)                         # (B, 1, 512, 512)

# instantiate models
model_t = UNet(num_classes=1).to(device)
model_s = StudentUNet(num_classes=1).to(device)

In [ ]:
# CELL
# DEFINING LOSSES

import torch
import torch.nn as nn
import torch.nn.functional as F

class SoftKDLoss(nn.Module):
    def __init__(self, T=3.0):
        super().__init__()
        self.T = float(T)

        assert T > 0, "temperature must be positive"

    def forward(self, logits_s, logits_t):
        T = self.T

        # compute teacher probs from teacher logits
        probs_t = torch.sigmoid(logits_t.detach() / T)

        # BCE between teacher and student logits
        assert logits_s.shape == probs_t.shape, "teacher and student logits must have the same shape"
        soft_loss = F.binary_cross_entropy_with_logits(
            logits_s / T,
            probs_t # torch.nn.functional.binary_cross_entropy_with_logits wants target with sigmoid already applied to it
        )

        return soft_loss * T**2

class TotalKDLoss(nn.Module):
    def __init__(
        self,
        hard_loss,
        soft_loss,
        w_soft=0.1
    ):
        super().__init__()
        self.hard_loss = hard_loss
        self.soft_loss = soft_loss
        self.w_soft = float(w_soft)
        self.w_hard = (1 - self.w_soft)

        assert 0.0 <= self.w_soft < 1.0, "w_soft must be [0, 1["

    def forward(self, logits_s, logits_t, gt):
        # compute hard and soft losses
        hard_loss = self.hard_loss(logits_s, gt)
        soft_loss = self.soft_loss(logits_s, logits_t)

        # compute total loss
        total = self.w_hard * hard_loss + self.w_soft * soft_loss

        # return
        return total, {
            "soft_loss": soft_loss.detach(),
            "hard_loss": hard_loss.detach()
        }

# ----------- CHOOSE SETTINGS AND INSTANTIATE

# instantiate hard KD loss
hard_loss = FocalBCEDiceLoss(
    dice_weight=1.0,
    alpha=0.25,
    gamma=2.0,
    neg_ohem_weight=0.05,
    neg_topk=1024
).to(device)

# instantiate soft KD loss
soft_loss = SoftKDLoss(
    T=3.0
).to(device)

# instantiate total loss
criterion_s = TotalKDLoss(
    hard_loss=hard_loss,
    soft_loss=soft_loss,
    w_soft=0.1, # w_soft + w_hard = 1.0
).to(device)

In [ ]:
# CELL
# KD TRAINING LOOP

import torch
from tqdm import tqdm

# ---------- STUDENT OPTIMIZER AND LR SCHEDULER

# define optimizer
optimizer_s = torch.optim.Adam(
    model_s.parameters(),
    lr=5e-5
)

# define scheduler
scheduler_s = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s,
    mode="min",
    factor=0.5,
    patience=2,
    threshold=1e-3,
    min_lr=5e-7
)

# ---------- TRAINING LOOP

# for each epoch...
def train_one_epoch(model_s, model_t, loader, criterion, optimizer, device):
    model_s.train()
    model_t.eval()

    running_hard_loss = 0.0
    running_soft_loss = 0.0
    running_total_loss = 0.0
    count_batches = 0

    # for each batch...
    for imgs, masks in tqdm(loader, desc="train", leave=False):
        # moving tensors to device
        imgs, masks = imgs.to(device), masks.to(device).float()

        # clear gradients
        optimizer.zero_grad()

        # teacher forward pass
        with torch.no_grad():
            logits_t = model_t(imgs, is_seg=True)

        # student forward pass
        logits_s = model_s(imgs)

        loss, parts = criterion(logits_s, logits_t, masks) # compute student loss
        loss.backward() # backpropagation
        optimizer.step() # update weights

        # bookkeeping
        running_hard_loss += parts["hard_loss"].item()
        running_soft_loss += parts["soft_loss"].item()
        running_total_loss += loss.item()
        count_batches += 1

    # return dict with avg train hard and soft losses in an epoch for logging
    return {
        "train_hard_loss": running_hard_loss / max(count_batches, 1),
        "train_soft_loss": running_soft_loss / max(count_batches, 1),
        "train_total_loss": running_total_loss / max(count_batches, 1)
    }

In [ ]:
# CELL
# VALIDATION LOOP

import torch
from tqdm import tqdm

# for each epoch...
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, threshold=0.5, eps=1e-6):
    model.eval()

    # loss bookkeeping
    running_loss = 0.0
    count_batches = 0

    # metrics bookkeeping
    dsc_sum = 0.0 # sum of DSC over positive samples
    iou_sum = 0.0 # sum of IoU over positive samples
    pos_count = 0 # number of positive samples evaluated

    neg_total = 0 # number of negative samples evaluated
    neg_fp = 0 # how many of those had false positives (thresholded)
    neg_fphw_sum = 0.0 # sum over negative samples of FP pixels/HW (thresholded)

    # for each batch...
    for imgs, masks in tqdm(loader, desc="val", leave=False):
        # moving tensors to device
        imgs, masks = imgs.to(device), masks.to(device).float()

        # forward pass
        logits = model(imgs)

        # compute loss
        loss = criterion(logits, masks)

        running_loss += loss.item()
        count_batches += 1

        # convert logits to binary predictions
        probs = torch.sigmoid(logits) # (B, 1, H, W)

        # apply threshold
        preds = (probs > threshold).float() # (B, 1, H, W)

        # identify pos/neg samples by gt
        is_pos = (masks.sum(dim=(1, 2, 3)) > 0) # bool tensor (B,)
        is_neg = ~is_pos # bool tensor (B,)

        # only compute DSC and IoU if there is at least one positive image in the batch
        if is_pos.any():
            # selecting positive samples
            p = preds[is_pos] # (B_pos, 1, H, W)
            g = masks[is_pos] # (B_pos, 1, H, W)
            # flattening
            p = p.flatten(start_dim=1) # (B_pos, H*W)
            g = g.flatten(start_dim=1) # (B_pos, H*W)

            # compute intersection per sample
            intersection = (p * g).sum(dim=1) # (B_pos,), TP pixels
            # compute sums per sample
            p_sum = p.sum(dim=1) # (B_pos,), predicted foreground pixels
            g_sum = g.sum(dim=1) # (B_pos,), GT foreground pixels
            # compute union per sample
            union = (p + g - p * g).sum(dim=1) # (B_pos), TP + FP + FN

            # compute DSC
            dsc = (2.0 * intersection + eps) / (p_sum + g_sum + eps) # (B_pos,)
            # compute IoU
            iou = (intersection + eps) / (union + eps) # (B_pos)

            # aggregate
            dsc_sum += dsc.sum().item()
            iou_sum += iou.sum().item()
            pos_count += dsc.numel()

        # compute FPIR and FP/HW
        if is_neg.any():
            preds_neg = preds[is_neg]

            # ----------------- FPIR ON NEGATIVES ONLY

            # add number of negatives samples in the batch to bookkeping
            neg_total += is_neg.sum().item()
            # add number of false positives to bookkeeping
            neg_fp += (preds_neg.sum(dim=(1,2,3)) > 0).sum().item()

            # ----------------- FP/HW ON NEGATIVS ONLY

            # flatten
            neg_frac = preds_neg.flatten(start_dim=1) # (B_neg, H*W)
            # mean over H*W, total pixels
            neg_frac = neg_frac.mean(dim=1) # (B_neg,)
            # add to bookkeeping
            neg_fphw_sum += neg_frac.sum().item()

    # define metrics per epoch
    val_loss = running_loss / max(count_batches, 1)
    val_dsc = dsc_sum / max(pos_count, 1)
    val_iou = iou_sum / max(pos_count, 1)

    val_fpir = neg_fp / max(neg_total, 1)
    val_fphw = neg_fphw_sum / max(neg_total, 1)

    # return a dictionary with validation loss and every metric
    return {
        "val_loss": val_loss,
        "val_dsc": val_dsc,
        "val_iou": val_iou,
        "val_fpir": val_fpir,
        "val_fphw": val_fphw
    }

In [ ]:
# CELL 14
# ORCHESTRATING KD RUN

from pathlib import Path
from datetime import datetime
import torch

# load trained teacher checkpoint
T_CKPT_FILE_NAME = "epoch_005_round_003_1770036394.pt"
T_CKPT_DIR = f"/content/drive/MyDrive/{my_drive_folder}/trained_teacher_model"
T_CKPT_PATH = f"{T_CKPT_DIR}/{T_CKPT_FILE_NAME}"
# load teacher checkpoint
trained_teacher_ckpt = torch.load(T_CKPT_PATH, map_location=device)
model_t.load_state_dict(trained_teacher_ckpt["model_state_dict"], strict=True)

model_t.eval()
for p in model_t.parameters():
    p.requires_grad_(False)

# creating checkpoint dir
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
CKPT_DIR = Path(f"/content/drive/MyDrive/{my_drive_folder}/checkpoints/{timestamp}_student_run")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint directory created at", CKPT_DIR)

# choose epoch count
"""
important note: warmup for OHEM takes 11 epochs to fully take place
"""
num_epochs = 7

# threshold sweep
thresholds = [0.25, 0.5, 0.7]
assert len(thresholds) > 0

train_hard_losses = []
val_losses = []

for epoch in range(num_epochs):
    # OHEM warmup schedule
    if epoch <= 5:
        hard_loss.neg_ohem_weight = 0.0
    elif epoch <= 7:
        hard_loss.neg_ohem_weight = 0.01
    elif epoch <= 9:
        hard_loss.neg_ohem_weight = 0.035
    else:
        hard_loss.neg_ohem_weight = 0.05

    # run training
    train_loss = train_one_epoch(
        model_s,
        model_t,
        train_loader,
        criterion_s,
        optimizer_s,
        device
    )
    train_total_loss = train_loss["train_total_loss"]
    train_hard_loss = train_loss["train_hard_loss"]
    train_soft_loss = train_loss["train_soft_loss"]

    # append training loss to list
    train_hard_losses.append(train_loss["train_hard_loss"])

    for t in thresholds:
        # run validation
        val_results = validate_one_epoch(model_s, val_loader, hard_loss, device, threshold=t)

        if t == thresholds[0]:
            # get the val loss for that epoch
            # note: any threshold is fine, but only one must be chosen; I chose the first
            val_loss = val_results["val_loss"]

            # append loss
            val_losses.append(val_loss)

            # step LR scheduler
            scheduler_s.step(train_hard_loss)

            # print epoch # and losses (printed once every epoch)
            print(
                "-------\n"
                f"EPOCH {epoch+1}/{num_epochs} | "
                f"TRAIN LOSS (H, S, T)={train_hard_loss:.4f}, {train_soft_loss:.4f}, {train_total_loss:.4f} | "
                f"VAL LOSS={val_results['val_loss']:.4f} | "
                f"LR={optimizer_s.param_groups[0]['lr']} | "
                f"OHEM WEIGHT={hard_loss.neg_ohem_weight}"
            )

        # print metrics (printed once per threshold value every epoch)
        print(
            f"THRESHOLD={t}\n"
            f"DSC={val_results['val_dsc']:.4f} | "
            f"IoU={val_results['val_iou']:.4f} | "
            f"FPIR={val_results['val_fpir']:.4f} | "
            f"FPHW={val_results['val_fphw']:.6f}"
        )

    # save basic checkpoints every 5 epochs
    if (epoch + 1) % 4 == 0:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model_s.state_dict(),
            "optimizer_state_dict": optimizer_s.state_dict(),
            "val_metrics": val_results,
        },
        CKPT_DIR / f"epoch_{epoch+1:03d}.pt")

# ----------------- PLOTTING EPOCH LOSS CURVES

import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_hard_losses, label="train_hard")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()